<a href="https://colab.research.google.com/github/nicolaiberk/llm_ws/blob/main/notebooks/05a_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Annotation with Generative Models

Today, we are going to see how to generate text and annotations with generative LLMs.

> ❗ ACTIVATE THE GPU BY SELECTING RUNTIME IN THE UPPER RIGHT > CONNECT TO RUNTIME > T4 GPU

In [ ]:
  !pip install transformers accelerate datasets evaluate sentence-transformers

> ❗ RESTART THE NOTEBOOK (DROPDOWN NEXT TO RUN ALL > RESTART SESSION)

## What happens in `pipeline()`?

*This section is based on [Chapter 2](https://huggingface.co/learn/llm-course/chapter2/2) of the 🤗 LLM Course.*

You are already familiar with the `pipeline()` method from our preceding notebook. Here is an example to jog your memory:

In [1]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier(
    [
        "My research is going to be whole different ballgame using LLMs!",
        "I hate this so much!",
    ]
)

/Users/niberk/Dropbox (Personal)/Teaching/LLM_WS/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9055778980255127},
 {'label': 'NEGATIVE', 'score': 0.9966409206390381}]

Alright, so this little function takes text inputs and transforms them into label predictions. But since transformers cannot simply be fit on the raw texts, several things have to happen in the pipeline:

![Visualization from Chapter 2 of teh Huggingface Course](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)

You already know from our last session how to load tokenizers of specific models and tokenize input texts using `AutoTokenizer()`. Lets's repeat this briefly:

In [2]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english" # this is the default model for sentiment-analysis
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [3]:
raw_inputs = [
    "My research is going to be whole different ballgame using LLMs!",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(inputs)

{'input_ids': tensor([[  101,  2026,  2470,  2003,  2183,  2000,  2022,  2878,  2367,  3608,
         16650,  2478,  2222,  5244,   999,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


*Can you explain what the padding and truncation parameters do here? What would the data look like if you would not specify them?*

Great. So we've already covered the first step! Let's look at how we could load a specific model and generate output embeddings of each text using our token IDs. We start by loading the model from the hub using `AutoModel()`:

In [4]:
from transformers import AutoModel

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint)

This architecture contains only the base Transformer module: given some inputs, it returns the output embeddings or *hidden states* of the model, which might then be fed into a classification head downstream. We can simply use the assigned object to run the forward pass and return the model embeddings:

In [5]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

torch.Size([2, 16, 768])


*Interpret these dimensions. What does each number describe?*

Now we need to transform these representations into useful classifications. As you heard already earlier today, models 'translate' embeddings into classifications using a **classification head**. Classification heads exist for many different tasks, e.g. question answering or token classification. We are interested in classifying our entire sequences. To do so, we load the model with a classification head for sequence classification using `AutoModelForSequenceClassification` instead of simply `AutoModel`.

In [6]:
from transformers import AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
outputs = model(**inputs)

In [7]:
print(outputs.logits.shape)

torch.Size([2, 2])


Instead of outputting the embeddings, this model contains a classification head which transforms the original, high-dimensional representation into a vector containing two values (one for each label). The values we get as output from our model don’t necessarily make sense by themselves - they are so-called logits. We can transform them to values bounded between 0 and 1 using the `softmax` function you already saw earlier today.

In [8]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
print(predictions.round(decimals=3))

tensor([[0.0940, 0.9060],
        [0.9990, 0.0010]], grad_fn=<RoundBackward1>)


To see which label corresponds to which prediction, you can check the model configuration:

In [9]:
model.config.id2label

{0: 'NEGATIVE', 1: 'POSITIVE'}

## Working with 🤗 `datasets`

Before we get started with training our own models, we need to understand how to provide useful inputs to our models. Huggingface provides a library called `datasets` to make our life a little easier.

The most basic function in the datasets library is probably `load_dataset`. Using this function, we can load an annotated dataset of synthetically generated phone calls, annotated for whether a given conversation is a scam or not.

In [10]:
from datasets import load_dataset

dataset = load_dataset("BothBosu/scam-dialogue")

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 1280
    })
    test: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 320
    })
})

As you can see, we get a dictionary (specifically, a `DatasetDict`) containing two parts of this dataset: a train set (referred to as a 'split) in the datasets lingo; and a test set. You can directly dowload either by specifying the split you are interested in:

In [12]:
dataset_train = load_dataset("BothBosu/scam-dialogue", split='train')

In [13]:
dataset_train

Dataset({
    features: ['dialogue', 'type', 'label'],
    num_rows: 1280
})

We have three features: dialogue (containing the text of the conversation), the type of the conversation (which we will ignore here), and the label, that is the outcome of interest - whether the conversation was a scam call (1) or not (0). *You can check the [dataset card](https://huggingface.co/datasets/BothBosu/scam-dialogue) for more information.*

You can inspect observations in the data simply by subsetting to the row of interest:

In [14]:
dataset['train'][0]

{'dialogue': "caller: Hello, is this John? receiver: Yes, it is. Who's calling? caller: My name is Officer Johnson from the Social Security Administration. How are you today? receiver: I'm fine, thank you. What can I do for you? caller: We've been trying to reach you about a very important matter. Your social security number has been compromised and we need to take immediate action to protect your identity. receiver: Oh no, what happened? caller: We've received reports of suspicious activity on your account and we need to verify some information to ensure your benefits aren't interrupted. receiver: Okay, what do you need to know? caller: Can you please confirm your social security number for me? receiver: Wait, I'm not comfortable giving that out over the phone. Is this really the Social Security Administration? caller: Ma'am, I assure you this is a legitimate call. We're trying to help you. If you don't cooperate, we'll have to suspend your benefits. receiver: I don't think so. I'm go

Another neat function is `train_test_split`: it divides the dataset into a train and a test set. We can use it here to subdivide our trainset into a validation and a training set. It is useful to set a seed for these operations, in order to make the process reproducible.

In [15]:
train_val_set = dataset['train'].train_test_split(test_size=0.1, shuffle=True, seed=42)

In [16]:
dataset['train'] = train_val_set['train']
dataset['validation'] = train_val_set['test']

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 1152
    })
    test: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 320
    })
    validation: Dataset({
        features: ['dialogue', 'type', 'label'],
        num_rows: 128
    })
})

Let's prepare the dataset. The datasets library offers a neat function called `map` to simply and quickly apply any transformation to our data. We start by applying some simple text cleaning, removing any weird punctuation and lowercasing everything (generally, this is usually not necessary with transformers - you should however remove HTML code, URLs, or other unnecessary parts of the text [unless you are interested in these features]).

In [18]:
import re

def clean_text(example):
    text = example["dialogue"].lower()  # lowercase
    text = re.sub(r"[^a-z0-9\s.,!?;:'\"-]", "", text)  # remove weird chars
    text = re.sub(r"\s+", " ", text).strip()  # normalize spaces
    example["dialogue"] = text
    return example

# Apply with map
dataset = dataset.map(clean_text)

Map: 100%|██████████| 128/128 [00:00<00:00, 7457.89 examples/s]


In [19]:
dataset['train'][0]

{'dialogue': "caller: hi, my name is john and i'm calling from xyz insurance company. how are you today? receiver: i'm doing great, thanks for asking. what can i do for you, john? caller: we're offering a special promotion on our life insurance policies and i was wondering if you'd be interested in hearing more about it. receiver: okay, sure. but before you start, can you tell me what makes this promotion so special? caller: well, we're offering a discounted rate for new customers who sign up within the next two weeks. it's a limited time offer. receiver: that sounds interesting. can you provide me with your company's license number and a physical address where i can verify your credentials? caller: absolutely. our license number is 123456 and our office is located at 123 main street, anytown usa. receiver: okay, let me just verify that real quick. yeah, everything checks out. you know, i've had some bad experiences with scammers posing as insurance companies in the past. caller: i com

Next step: tokenization! We load the tokenizer of a pre-trained bert model and define the function we want to apply:

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def encode(examples):
    return tokenizer(examples["dialogue"], truncation=True, padding="max_length")


We can then use `map` to encode each dataset in the dictionary:

In [21]:
dataset = dataset.map(encode, batched=True)
dataset

Map: 100%|██████████| 128/128 [00:00<00:00, 1447.14 examples/s]


DatasetDict({
    train: Dataset({
        features: ['dialogue', 'type', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1152
    })
    test: Dataset({
        features: ['dialogue', 'type', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 320
    })
    validation: Dataset({
        features: ['dialogue', 'type', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 128
    })
})

As you can see, we have added three columns to our dataset, containing the tokenizer output for each conversation. Note that the logic of map allows you to apply any transformation you want - you simply need to change the supplied function.

Our model will expect a variable called 'labels' as outcome. So let's rename our variable:

In [22]:
dataset

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'type', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1152
    })
    test: Dataset({
        features: ['dialogue', 'type', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 320
    })
    validation: Dataset({
        features: ['dialogue', 'type', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 128
    })
})

In [23]:
dataset = dataset.rename_column("label", "labels")

Lastly, we select the columns of interest using `select_columns`.

*For a full overview of pre-processing functions, check out the [datasets documentation](https://huggingface.co/docs/datasets/en/process).*

In [24]:
dataset = dataset.select_columns(["input_ids", "token_type_ids", "attention_mask", "labels"])

*If you want to learn more about 🤗 `datasets` you can check out the [`datasets` tutorial](https://huggingface.co/docs/datasets/en/quickstart) and the [chapter on `datasets` in the 🤗 LLM course](https://huggingface.co/learn/llm-course/chapter5/1).*

### Using your own data as a dataset

datasets also offers helpful functions to load your own data. There are [many functions](https://huggingface.co/docs/datasets/en/loading) to do this, including loading directly from disk. However, especially if you need to prepare the dataset beforehand, it is usually simplest to work with `pandas` and then transform this dataframe to a datasets dataset:

In [25]:
from datasets import Dataset
import pandas as pd
df = pd.DataFrame({"a": [1, 2, 3]})
tmp_dataset = Dataset.from_pandas(df)

## Generating text with generative models

Finally, generative models! We will start by simply generating some text using a family of small generative models developed by huggingface.

### Simple Inference

The all-powerful `pipeline` is again the simplest way to get inference running quickly:

In [28]:
from transformers import pipeline

pipe = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M-Instruct")
messages = [
    "Now this is a story all about how",
]
pipe(messages)

Device set to use mps:0


[[{'generated_text': 'Now this is a story all about how the human mind will be changed by the memory of the past.  I\'m not sure about the moral implications of this story.  I\'d love your thoughts on the story.\n\nIn "The Memory Keeper," a young woman named Aria discovers a mysterious box that contains memories of an ancient civilization.  The box is said to hold the secrets of the past and Aria is drawn into a world of magic, politics, and love.  As Aria delves deeper into the box, she begins to realize that the memories are not just simple recollections - they are the memories of a past that is not her own.  The memories take on a life of their own and Aria finds herself trapped in a world of her own making.\n\nOne of the most intriguing aspects of "The Memory Keeper" is its exploration of the power of memories to shape our understanding of ourselves.  As Aria navigates this strange and fascinating world, she is forced to confront the possibility that the memories she has taken from

### Chat

Many applications of LLMs require a chat template. We can use the tokenizer to enforce this template. The template simply indicates which parts of the text are from to the user and which are/should be from the assistant.

Remember: LLM chats are just roleplay with special tokens!

In [29]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct", padding_side='left')
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")

In [30]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=False)
tokenized_chat

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWho are you?<|im_end|>\n'

In this context, it is helpful to add the generation prompt indicating that the text should be generated in the role of the assistant. Otherwise, the model might generate more text as the user instead ([more](https://huggingface.co/docs/transformers/en/chat_templating?template=Mistral#addgenerationprompt)).

In [31]:
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [32]:
tokenized_chat

'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWho are you?<|im_end|>\n<|im_start|>assistant\n'

In [33]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	return_dict=True, # retains attention mask
	return_tensors="pt", # returns tensors
).to(model.device) # more efficient to put on device

In [34]:
inputs

{'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198, 10576,   359,   346,    47,     2,   198,
             1,   520,  9531,   198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [35]:
outputs = model.generate(**inputs, max_new_tokens=100) # note the max_new_tokens parameter
print(tokenizer.decode(outputs[0])) # note that the entire conversation is returned, including the system prompt.

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Who are you?<|im_end|>
<|im_start|>assistant
I'm a chatbot designed to assist users with their language learning needs. I was trained on a vast amount of text data, including various languages, grammar rules, and vocabulary. I can help with language learning, grammar, and vocabulary exercises, as well as provide explanations for various language concepts.<|im_end|>


### Zero-shot prompting

In order to get proper annotations from our model, we can simply ask the model to generate the relevant outputs. This is as simple as writing a prompt. Remember the best practices we discussed earlier today.

In [36]:
messages = [
    [{"role": "user", "content": """You are an expert annotator with years of experience annotating social science data.
    Your main task is to annotate whether the following text is about AI or not.
    Respond ONLY with the label "AI" or "NOT AI". Do NOT provide an explanation

    Text: "SmolLM is a pretty impressive model!"
    """}
    ],
    [{"role": "user", "content": """You are an expert annotator with years of experience annotating social science data.
    Your main task is to annotate whether the following text is about AI or not.
    Respond ONLY with the label "AI" or "NOT AI". Do NOT provide an explanation

    Text: "The weather is horrible today!"
    """}]
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
  padding=True,
	return_dict=True, # retains attention mask
	return_tensors="pt", # returns tensors
).to(model.device) # more efficient to put on device

In [37]:
messages

[[{'role': 'user',
   'content': 'You are an expert annotator with years of experience annotating social science data.\n    Your main task is to annotate whether the following text is about AI or not.\n    Respond ONLY with the label "AI" or "NOT AI". Do NOT provide an explanation\n\n    Text: "SmolLM is a pretty impressive model!"\n    '}],
 [{'role': 'user',
   'content': 'You are an expert annotator with years of experience annotating social science data.\n    Your main task is to annotate whether the following text is about AI or not.\n    Respond ONLY with the label "AI" or "NOT AI". Do NOT provide an explanation\n\n    Text: "The weather is horrible today!"\n    '}]]

In [38]:
inputs

{'input_ids': tensor([[    1,  9690,   198,  2683,   359,   253,  5356,  5646, 11173,  3365,
          3511,   308, 34519,    28,  7018,   411,   407, 19712,  8182,     2,
           198,     1,  4093,   198,  2683,   359,   354,  4507, 13666,  1508,
           351,   929,   282,  1786, 13666,   674,  1329,  2092,   940,    30,
           472,  2789,  1085,  3856,   314,   288, 13666,   368,  1991,   260,
          1695,  1694,   314,   563,  5646,   355,   441,    30,   472, 30417,
         44339,   351,   260,  4368,   476, 13701,    18,   355,   476, 18083,
          5646,  2227,  3315,  9695,  1538,   354,  7718,  1004,  9378,    42,
           476,  9207,   308, 34519,   314,   253,  5740, 10402,  1743,  6653,
          2367,     2,   198,     1,   520,  9531,   198],
        [    2,     2,     2,     1,  9690,   198,  2683,   359,   253,  5356,
          5646, 11173,  3365,  3511,   308, 34519,    28,  7018,   411,   407,
         19712,  8182,     2,   198,     1,  4093,   198, 

In [39]:
outputs = model.generate(**inputs, max_new_tokens=3)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:]))
print(tokenizer.decode(outputs[1][inputs['input_ids'].shape[1]:]))

AI<|im_end|><|im_end|>
NOT AI<|im_end|>


### Few-shot

In [40]:
messages = [
    [{"role": "user", "content": """You are an expert annotator with years of experience annotating social science data.
    Your main task is to annotate whether the following text is about artificial intelligence or not.
    Respond ONLY with the label "AI" or "NOT AI". Do NOT provide an explanation

    Example:
    "SmolLM is a pretty impressive model!"
    """},
     {"role": "assistant", "content": "AI"},
     {"role": "user", "content": """
    Example:
    "The weather is horrible today!"
    """},
     {"role": "assistant", "content": "NOT AI"},
     {"role": "user", "content": """
     Text: "The impact of the new wave on automation on the labour market is not yet clear."
     """}
    ]]

In [41]:
tokenized_chat = tokenizer.apply_chat_template(messages, tokenize=False)
tokenized_chat

['<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nYou are an expert annotator with years of experience annotating social science data.\n    Your main task is to annotate whether the following text is about artificial intelligence or not.\n    Respond ONLY with the label "AI" or "NOT AI". Do NOT provide an explanation\n\n    Example:\n    "SmolLM is a pretty impressive model!"\n    <|im_end|>\n<|im_start|>assistant\nAI<|im_end|>\n<|im_start|>user\n\n    Example:\n    "The weather is horrible today!"\n    <|im_end|>\n<|im_start|>assistant\nNOT AI<|im_end|>\n<|im_start|>user\n\n     Text: "The impact of the new wave on automation on the labour market is not yet clear."\n     <|im_end|>\n']

In [42]:
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
  padding=True,
	return_dict=True, # retains attention mask
	return_tensors="pt", # returns tensors
).to(model.device) # more efficient to put on device

In [43]:
outputs = model.generate(**inputs, max_new_tokens=3)
print(tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:]))

NOT AI<|im_end|>


# Exercise

1. Think of a concept of interest for your research. Operationalize it with some labels. Define 2-3 example texts with the associated labels for each category. These texts will serve as the context explaining to our model how to annotate.

In [ ]:
# Define the context
example_texts = [
]

# Associated labels
example_labels = [
]

2. Create a query to pass to our model and an example text you wish to classify.

In [ ]:
query = """
"""

In [ ]:
article_to_classify = ""

3. Use the chat template to create a structured prompt to provide to the model.

In [ ]:
chat_template = [
    ]

In [ ]:
chat_template

4. Generate an annotation from the model. Are you happy with the model's annotation?

In [58]:
## model definition
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct", padding_side='left')
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")

In [ ]:
## tokenization
inputs = 

In [ ]:
## inference
outputs = model.